In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
        .appName("Bronze")
        .config("spark.driver.memory", "6g")
        .config("spark.executor.memory", "6g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import logging
from datetime import datetime

In [3]:
raw_path = r"../00_storage_raw/yellow_taxi"
bronze_path = r"../01_storage_bronze/yellow_taxi"
quarantine_path = r"../01_storage_bronze/yellow_taxi_quarantine"
tracking_path = r"../01_storage_bronze/_metadata_processed_files"

In [4]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(log_dir, "01_raw_to_bronze.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [6]:
total_count = spark.read.parquet(raw_path).count()
print(f"Total records in landing zone: {total_count}")

Total records in landing zone: 59799808


In [7]:
tracking_schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("processed_time", TimestampType(), True)
])

if not os.path.exists(tracking_path):
    empty_df = spark.createDataFrame([], tracking_schema)
    empty_df.write.format("delta").save(tracking_path)  

### Schema Enforcement

In [8]:
processed_df = spark.read.format("delta").load(tracking_path)

processed_files = set(
    row.file_name for row in processed_df.select("file_name").collect()
)


In [9]:
from pyspark.sql.functions import max

latest = processed_df.select(max("processed_time")).collect()[0][0]

print(f"Latest processed time: {latest}")

Latest processed time: None


In [10]:
mySchema = StructType().fromDDL("""
    VendorID BIGINT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    RatecodeID BIGINT,
    store_and_fwd_flag STRING,
    PULocationID BIGINT,
    DOLocationID BIGINT,
    payment_type BIGINT,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    airport_fee DOUBLE,
    year INT
""")

In [11]:
def bronze_transformations(df_trip_data):

    # Fix schema drift
    for field in mySchema.fields:
        if field.name in df_trip_data.columns:
            df_trip_data = df_trip_data.withColumn(
                field.name,
                col(field.name).cast(field.dataType)
            )

    # Handle column case mismatch
    if "Airport_fee" in df_trip_data.columns:
        df_trip_data = df_trip_data.withColumnRenamed("Airport_fee", "airport_fee")

    # Metadata columns
    df_trip_data = df_trip_data \
        .withColumn("bronze_ingestion_time", current_timestamp()) \
        .withColumn("source_file", input_file_name())

    # Partition column
    df_trip_data = df_trip_data.withColumn(
        "year", 
        year(col("tpep_dropoff_datetime"))
    )

    # GOOD DATA
    good_df = df_trip_data.filter(
        (col("year") >= 2025) & (col("year") <= 2026) & col("year").isNotNull()
    )

    # BAD DATA
    bad_df = df_trip_data.filter(
        (col("year") < 2025) | (col("year") > 2026) | col("year").isNull()
    )

    # -----------------------------------
    # DEDUPLICATION SAFETY
    # -----------------------------------
    good_df = good_df.dropDuplicates([
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime"
    ])

    # -----------------------------------
    # WRITE BRONZE
    # -----------------------------------
    good_df \
        .repartition(3, "year") \
        .write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("year") \
        .save(bronze_path)

    # -----------------------------------
    # WRITE QUARANTINE
    # -----------------------------------
    bad_df \
        .repartition(1) \
        .write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("year") \
        .save(quarantine_path)

In [12]:
new_files = []

for root, dirs, files in os.walk(raw_path):
    for file in files:
        if file.endswith(".parquet"):
            full_path = os.path.join(root, file)

            # SKIP ALREADY PROCESSED FILES
            if full_path in processed_files:
                continue

            logging.info(f"Processing NEW file {full_path}")

            try:
                df_trip_data = spark.read.parquet(full_path)

                bronze_transformations(df_trip_data)

                new_files.append((full_path, datetime.now()))
                logging.info(f"Successfully processed {full_path}")

            except Exception as e:
                logging.exception(f"Error processing {full_path}: {e}")

In [13]:
# -----------------------------------
# UPDATE TRACKING TABLE
# -----------------------------------
if new_files:
    new_files_df = spark.createDataFrame(
        new_files,
        ["file_name", "processed_time"]
    )

    new_files_df.write \
        .format("delta") \
        .mode("append") \
        .save(tracking_path)

logging.info("Bronze load completed")